---
layout: post
comments: false
title: Parallel and Distributed Computing
description:  This lesson will introduce the concepts of parallel and distributed computing, including how to design algorithms that can be executed in parallel and how to manage communication and synchronization between processes.
permalink: /csp/parallel-distributed/
author: MalwareMadness
codemirror: true
---

# Parallel and Distributed Computing

<div style="background:#0f172a; border-left:4px solid #3b82f6; border-radius:0 8px 8px 0; padding:14px 18px; margin:16px 0; font-size:0.95em; color:#cbd5e1;"><strong style="color:#3b82f6;">The Problem.</strong>&ensp;You have 1,000 images to compress. One machine handles one at a time and takes 10 ms each — that's <strong>10 seconds total</strong> if you're clever, or <strong>10 minutes</strong> if you're not. The difference is <em>strategy</em>.</div>

Each section explains one strategy and lets you run real Python to see it in action. **Check your understanding at the bottom once you've read all three.**

<div style="border-left:4px solid #f59e0b; padding:4px 0 4px 14px; margin:28px 0 10px;"><span style="color:#f59e0b; font-size:1.25em; font-weight:800;">🟡 Serial Computing</span><span style="color:#64748b; font-size:0.82em; margin-left:10px;">1 processor · tasks run one after another</span></div>

A single processor handles one task at a time. Task 2 cannot start until Task 1 finishes.
**Total time = sum of every task.** Simple and predictable — but it doesn't scale.

<div style="overflow-x:auto; margin:12px 0;"><div class="mermaid">
%%{init: {"theme": "base", "themeVariables": {"fontSize": "15px"}, "flowchart": {"nodeSpacing": 50, "rankSpacing": 60, "htmlLabels": true}}}%%
graph LR
    T1["⏳ Task 1"] -->|done| T2["⏳ Task 2"] -->|done| T3["⏳ Task 3"] -->|done| T4["⏳ Task 4"] -->|done| R(["✅ Result"])
</div></div>

{% capture challenge_ser %}
Run serial processing and observe the timing. Try changing the unit values — notice total time always equals the sum.
{% endcapture %}

{% capture code_ser %}
import time

def work(units):
    return sum(i * i for i in range(units * 25_000))

tasks = [("Task A", 2), ("Task B", 4), ("Task C", 1), ("Task D", 3)]

print("SERIAL -- one processor, one task at a time")
print("=" * 44)
wall = time.perf_counter()
for name, units in tasks:
    t0 = time.perf_counter()
    work(units)
    print(f"  {name}  ->  {(time.perf_counter()-t0)*1000:.1f} ms")
total = (time.perf_counter() - wall) * 1000
print("=" * 44)
print(f"  Total: {total:.1f} ms  <- sum of every task")
print()
print("Increase any unit -- notice total always grows.")
{% endcapture %}

{% include runners/code.html
   runner_id="ser-demo"
   language="python"
   challenge=challenge_ser
   code=code_ser
   height="340px"
%}

<div style="border-left:4px solid #3b82f6; padding:4px 0 4px 14px; margin:28px 0 10px;"><span style="color:#3b82f6; font-size:1.25em; font-weight:800;">🔵 Parallel Computing</span><span style="color:#64748b; font-size:0.82em; margin-left:10px;">4+ cores · same machine · shared memory · no network cost</span></div>

Multiple cores inside the **same machine** work on different tasks simultaneously.
Because cores share memory directly there is zero network overhead.
**Total time ≈ slowest single task** — not the sum.

<div style="overflow-x:auto; margin:12px 0;"><div class="mermaid">
%%{init: {"theme": "base", "themeVariables": {"fontSize": "15px"}, "flowchart": {"nodeSpacing": 50, "rankSpacing": 60, "htmlLabels": true}}}%%
graph TD
    Q(["📋 Task Queue"]) --> C1["🖥️ Core 1<br/>Task 1"]
    Q --> C2["🖥️ Core 2<br/>Task 2"]
    Q --> C3["🖥️ Core 3<br/>Task 3"]
    Q --> C4["🖥️ Core 4<br/>Task 4"]
    C1 --> R(["✅ Result"])
    C2 --> R
    C3 --> R
    C4 --> R
</div></div>

{% capture challenge_par %}
Run parallel processing with ThreadPoolExecutor. Try making Task B much heavier — notice other tasks finish while it runs.
{% endcapture %}

{% capture code_par %}
import time
from concurrent.futures import ThreadPoolExecutor

def work(args):
    name, units = args
    result = sum(i * i for i in range(units * 25_000))
    return name

tasks = [("Task A", 2), ("Task B", 4), ("Task C", 1), ("Task D", 3)]

print("PARALLEL -- 4 workers, same machine")
print("=" * 44)
wall = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    done = list(pool.map(work, tasks))
for name in done:
    print(f"  {name}  ->  done")
total = (time.perf_counter() - wall) * 1000
print("=" * 44)
print(f"  Total: {total:.1f} ms")
print()
print("Total = slowest task, not the sum.")
print("Try making Task B much larger -- others finish while it runs.")
{% endcapture %}

{% include runners/code.html
   runner_id="par-demo"
   language="python"
   challenge=challenge_par
   code=code_par
   height="340px"
%}

<div style="border-left:4px solid #22c55e; padding:4px 0 4px 14px; margin:28px 0 10px;"><span style="color:#22c55e; font-size:1.25em; font-weight:800;">🟢 Distributed Computing</span><span style="color:#64748b; font-size:0.82em; margin-left:10px;">many machines · network overhead · virtually unlimited scale</span></div>

Separate machines — called **nodes** — collaborate over a network.
Each node works independently, then sends its result back.
The network handshake adds latency per task, but you gain scale no single machine can match.
Distributed is essential when the problem is **too large for one machine**.

<div style="overflow-x:auto; margin:12px 0;"><div class="mermaid">
%%{init: {"theme": "base", "themeVariables": {"fontSize": "15px"}, "flowchart": {"nodeSpacing": 50, "rankSpacing": 60, "htmlLabels": true}}}%%
graph TD
    Q(["📋 Task Queue"]) --> W1["⚙️ Node 1<br/>Task 1"]
    Q --> W2["⚙️ Node 2<br/>Task 2"]
    Q --> W3["⚙️ Node 3<br/>Task 3"]
    Q --> W4["⚙️ Node 4<br/>Task 4"]
    W1 -->|"📡 network"| R(["✅ Result"])
    W2 -->|"📡 network"| R
    W3 -->|"📡 network"| R
    W4 -->|"📡 network"| R
</div></div>

{% capture challenge_dis %}
Run the distributed simulation with asyncio. Try changing NETWORK_MS to 500 — watch overhead dominate when tasks are light.
{% endcapture %}

{% capture code_dis %}
import asyncio, time

NETWORK_MS = 60  # simulated handshake per node

async def remote_node(name, units):
    await asyncio.sleep(NETWORK_MS / 1000)
    result = sum(i * i for i in range(units * 25_000))
    return name

tasks = [("Task A", 2), ("Task B", 4), ("Task C", 1), ("Task D", 3)]

async def main():
    print(f"DISTRIBUTED -- 4 nodes, {NETWORK_MS}ms overhead each")
    print("=" * 50)
    wall = time.perf_counter()
    done = await asyncio.gather(*[remote_node(n, u) for n, u in tasks])
    for name in done:
        print(f"  {name}  ->  done  (remote node)")
    total = (time.perf_counter() - wall) * 1000
    print("=" * 50)
    print(f"  Total: {total:.1f} ms")
    print()
    print(f"Network delays ran concurrently: ~{NETWORK_MS}ms total, not {NETWORK_MS*4}ms.")
    print("Try NETWORK_MS = 500 to see when overhead dominates.")

asyncio.run(main())
{% endcapture %}

{% include runners/code.html
   runner_id="dis-demo"
   language="python"
   challenge=challenge_dis
   code=code_dis
   height="340px"
%}

## Check Your Understanding

Now that you've seen all three paradigms — answer without looking back!

<div style="background:#0f172a; border:1px solid #334155; border-radius:9px; padding:14px 16px; margin:14px 0;">
  <p style="color:#e2e8f0; margin:0 0 10px; font-weight:600; font-size:0.95em;">❓ A bakery has one oven and bakes one tray of cookies at a time. Tray 2 goes in only after tray 1 comes out. Which computing paradigm does this model?</p>
  <div style="display:flex; flex-direction:column; gap:6px;">
    <label id="q-ser-opt-a" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-ser" value="a" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Parallel — multiple trays bake simultaneously</label>
    <label id="q-ser-opt-b" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-ser" value="b" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Serial — one tray finishes before the next begins</label>
    <label id="q-ser-opt-c" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-ser" value="c" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Distributed — work is split across several bakeries</label>
  </div>
  <div style="margin-top:10px; display:flex; align-items:center; gap:10px;">
    <button onclick="pdcCheck('q-ser','b')" style="padding:5px 16px; border-radius:6px; border:none; background:#3b82f6; color:#fff; cursor:pointer; font-size:0.85em; font-weight:bold;">Check →</button>
    <span id="q-ser-fb" style="font-size:0.85em;"></span>
  </div>
  <div id="q-ser-exp" style="display:none; margin-top:8px; color:#94a3b8; font-size:0.82em; border-top:1px solid #1e293b; padding-top:8px;">One oven, one tray at a time — total baking time is the sum of every tray. Parallel would need multiple ovens; distributed would involve different bakery locations.</div>
</div>

<div style="background:#0f172a; border:1px solid #334155; border-radius:9px; padding:14px 16px; margin:14px 0;">
  <p style="color:#e2e8f0; margin:0 0 10px; font-weight:600; font-size:0.95em;">❓ You are rendering a 3D film. Every frame is completely independent, and your workstation has a 16-core CPU. Which strategy cuts render time most?</p>
  <div style="display:flex; flex-direction:column; gap:6px;">
    <label id="q-par-opt-a" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-par" value="a" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Serial — render frames one by one on a single core</label>
    <label id="q-par-opt-b" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-par" value="b" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Parallel — split frames across all 16 cores on the same machine</label>
    <label id="q-par-opt-c" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-par" value="c" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Distributed — send frames to machines in different data centers</label>
  </div>
  <div style="margin-top:10px; display:flex; align-items:center; gap:10px;">
    <button onclick="pdcCheck('q-par','b')" style="padding:5px 16px; border-radius:6px; border:none; background:#3b82f6; color:#fff; cursor:pointer; font-size:0.85em; font-weight:bold;">Check →</button>
    <span id="q-par-fb" style="font-size:0.85em;"></span>
  </div>
  <div id="q-par-exp" style="display:none; margin-top:8px; color:#94a3b8; font-size:0.82em; border-top:1px solid #1e293b; padding-top:8px;">Parallel: frames are independent, cores share memory, zero network cost. Distributed would also work but adds unnecessary overhead when one machine already has enough cores.</div>
</div>

<div style="background:#0f172a; border:1px solid #334155; border-radius:9px; padding:14px 16px; margin:14px 0;">
  <p style="color:#e2e8f0; margin:0 0 10px; font-weight:600; font-size:0.95em;">❓ Training a state-of-the-art AI model requires 40 TB of data — far more than the RAM of any single machine. Which approach is necessary?</p>
  <div style="display:flex; flex-direction:column; gap:6px;">
    <label id="q-dis-opt-a" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-dis" value="a" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Serial — process data chunks one at a time on one machine</label>
    <label id="q-dis-opt-b" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-dis" value="b" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Parallel — use every core on one powerful server</label>
    <label id="q-dis-opt-c" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-dis" value="c" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Distributed — spread data and work across many machines</label>
  </div>
  <div style="margin-top:10px; display:flex; align-items:center; gap:10px;">
    <button onclick="pdcCheck('q-dis','c')" style="padding:5px 16px; border-radius:6px; border:none; background:#3b82f6; color:#fff; cursor:pointer; font-size:0.85em; font-weight:bold;">Check →</button>
    <span id="q-dis-fb" style="font-size:0.85em;"></span>
  </div>
  <div id="q-dis-exp" style="display:none; margin-top:8px; color:#94a3b8; font-size:0.82em; border-top:1px solid #1e293b; padding-top:8px;">When the problem exceeds what a single machine can hold, distributed computing is the only viable option. Real AI training (GPT-4, Gemini) runs on thousands of distributed nodes.</div>
</div>

### Scenario Challenges

<div style="background:#0f172a; border:1px solid #334155; border-radius:9px; padding:14px 16px; margin:14px 0;">
  <p style="color:#e2e8f0; margin:0 0 10px; font-weight:600; font-size:0.95em;">❓ Scenario A — You have a 200 GB dataset to process on a machine with 12 cores. All tasks are fully independent.</p>
  <div style="display:flex; flex-direction:column; gap:6px;">
    <label id="q-sc1-opt-a" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc1" value="a" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Serial</label>
    <label id="q-sc1-opt-b" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc1" value="b" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Parallel</label>
    <label id="q-sc1-opt-c" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc1" value="c" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Distributed</label>
  </div>
  <div style="margin-top:10px; display:flex; align-items:center; gap:10px;">
    <button onclick="pdcCheck('q-sc1','b')" style="padding:5px 16px; border-radius:6px; border:none; background:#3b82f6; color:#fff; cursor:pointer; font-size:0.85em; font-weight:bold;">Check →</button>
    <span id="q-sc1-fb" style="font-size:0.85em;"></span>
  </div>
  <div id="q-sc1-exp" style="display:none; margin-top:8px; color:#94a3b8; font-size:0.82em; border-top:1px solid #1e293b; padding-top:8px;">Parallel: data fits on one machine, tasks are independent, 12 cores give near 12× speedup with zero network overhead.</div>
</div>

<div style="background:#0f172a; border:1px solid #334155; border-radius:9px; padding:14px 16px; margin:14px 0;">
  <p style="color:#e2e8f0; margin:0 0 10px; font-weight:600; font-size:0.95em;">❓ Scenario B — A genomics pipeline must search a 500 TB DNA database. The largest single server available has 2 TB of RAM.</p>
  <div style="display:flex; flex-direction:column; gap:6px;">
    <label id="q-sc2-opt-a" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc2" value="a" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Serial</label>
    <label id="q-sc2-opt-b" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc2" value="b" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Parallel</label>
    <label id="q-sc2-opt-c" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc2" value="c" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Distributed</label>
  </div>
  <div style="margin-top:10px; display:flex; align-items:center; gap:10px;">
    <button onclick="pdcCheck('q-sc2','c')" style="padding:5px 16px; border-radius:6px; border:none; background:#3b82f6; color:#fff; cursor:pointer; font-size:0.85em; font-weight:bold;">Check →</button>
    <span id="q-sc2-fb" style="font-size:0.85em;"></span>
  </div>
  <div id="q-sc2-exp" style="display:none; margin-top:8px; color:#94a3b8; font-size:0.82em; border-top:1px solid #1e293b; padding-top:8px;">The database does not fit on one machine — distributed computing is required. Real genomics platforms partition data across hundreds of nodes.</div>
</div>

<div style="background:#0f172a; border:1px solid #334155; border-radius:9px; padding:14px 16px; margin:14px 0;">
  <p style="color:#e2e8f0; margin:0 0 10px; font-weight:600; font-size:0.95em;">❓ Scenario C — You need to run a single, indivisible 30-second calculation that cannot be broken into sub-tasks at all.</p>
  <div style="display:flex; flex-direction:column; gap:6px;">
    <label id="q-sc3-opt-a" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc3" value="a" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Serial</label>
    <label id="q-sc3-opt-b" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc3" value="b" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Parallel</label>
    <label id="q-sc3-opt-c" style="cursor:pointer; padding:9px 14px; border-radius:7px; border:1px solid #334155; color:#cbd5e1; background:#1e293b; display:flex; gap:10px; align-items:flex-start;"><input type="radio" name="q-sc3" value="c" style="margin-top:3px; accent-color:#3b82f6; flex-shrink:0;"> Distributed</label>
  </div>
  <div style="margin-top:10px; display:flex; align-items:center; gap:10px;">
    <button onclick="pdcCheck('q-sc3','a')" style="padding:5px 16px; border-radius:6px; border:none; background:#3b82f6; color:#fff; cursor:pointer; font-size:0.85em; font-weight:bold;">Check →</button>
    <span id="q-sc3-fb" style="font-size:0.85em;"></span>
  </div>
  <div id="q-sc3-exp" style="display:none; margin-top:8px; color:#94a3b8; font-size:0.82em; border-top:1px solid #1e293b; padding-top:8px;">If a task is truly indivisible, neither parallel nor distributed can speed it up. Amdahl's Law: speedup is bounded by the sequential fraction of work.</div>
</div>



### Scenario D — Code Challenge

The script below processes 8 images one at a time — it's too slow. **Convert the serial loop to run in parallel** using `ThreadPoolExecutor`. The built-in checker will measure your speedup and tell you if you passed.


<div style="margin:14px 0; display:flex; flex-direction:column; gap:6px;">

<details style="background:#0f172a; border:1px solid #334155; border-radius:8px; padding:10px 14px;">
<summary style="cursor:pointer; color:#f59e0b; font-weight:600; user-select:none; list-style:none;">
  💡 Hint 1 — What kind of problem is this?
</summary>
<div style="margin-top:10px; color:#cbd5e1; border-top:1px solid #1e293b; padding-top:10px; font-size:0.9em;">
  The 8 images are completely <strong>independent</strong> of each other — compressing image 3
  doesn't need any result from image 2. When tasks are independent, they're perfect candidates
  for <em>parallel</em> execution. You need a Python tool that can run the same function
  on multiple inputs at the same time.
</div>
</details>

<details style="background:#0f172a; border:1px solid #334155; border-radius:8px; padding:10px 14px;">
<summary style="cursor:pointer; color:#f59e0b; font-weight:600; user-select:none; list-style:none;">
  💡 Hint 2 — Which module and class?
</summary>
<div style="margin-top:10px; color:#cbd5e1; border-top:1px solid #1e293b; padding-top:10px; font-size:0.9em;">
  Python's <code>concurrent.futures</code> module is already imported at the top.
  The class you want is <strong><code>ThreadPoolExecutor</code></strong> — it manages a pool
  of worker threads and distributes tasks across them automatically.
  <pre style="background:#020617; padding:8px; border-radius:6px; margin-top:8px; color:#94a3b8; font-size:0.85em;">from concurrent.futures import ThreadPoolExecutor</pre>
</div>
</details>

<details style="background:#0f172a; border:1px solid #334155; border-radius:8px; padding:10px 14px;">
<summary style="cursor:pointer; color:#f59e0b; font-weight:600; user-select:none; list-style:none;">
  💡 Hint 3 — How do you create a pool?
</summary>
<div style="margin-top:10px; color:#cbd5e1; border-top:1px solid #1e293b; padding-top:10px; font-size:0.9em;">
  Use a <code>with</code> block so the pool is cleaned up automatically when done.
  <code>max_workers</code> controls how many threads run at once — set it to the number
  of tasks or your CPU core count, whichever is smaller.
  <pre style="background:#020617; padding:8px; border-radius:6px; margin-top:8px; color:#94a3b8; font-size:0.85em;">with ThreadPoolExecutor(max_workers=4) as pool:
    # submit work here</pre>
</div>
</details>

<details style="background:#0f172a; border:1px solid #334155; border-radius:8px; padding:10px 14px;">
<summary style="cursor:pointer; color:#f59e0b; font-weight:600; user-select:none; list-style:none;">
  💡 Hint 4 — How do you map tasks to the pool?
</summary>
<div style="margin-top:10px; color:#cbd5e1; border-top:1px solid #1e293b; padding-top:10px; font-size:0.9em;">
  <code>pool.map(fn, iterable)</code> works like the built-in <code>map()</code> —
  it calls <code>fn</code> once for each item in the iterable, but runs them in parallel
  across the worker threads. Wrap it in <code>list()</code> to collect all results.
  <pre style="background:#020617; padding:8px; border-radius:6px; margin-top:8px; color:#94a3b8; font-size:0.85em;">results = list(pool.map(compress_image, images))</pre>
  Notice that <code>compress_image</code> already takes a single argument (<code>image_id</code>),
  so it plugs straight into <code>pool.map</code> without any changes.
</div>
</details>

<details style="background:#0f172a; border:1px solid #334155; border-radius:8px; padding:10px 14px;">
<summary style="cursor:pointer; color:#f59e0b; font-weight:600; user-select:none; list-style:none;">
  💡 Hint 5 — What exactly needs to change?
</summary>
<div style="margin-top:10px; color:#cbd5e1; border-top:1px solid #1e293b; padding-top:10px; font-size:0.9em;">
  Only the lines between the two timing markers need to change. Replace the entire
  serial <code>for</code> loop (including the <code>results = []</code> line) with a
  single <code>with</code> block. Everything else — the function definition, the
  <code>images</code> list, and the checker — stays exactly as-is.
  <pre style="background:#020617; padding:8px; border-radius:6px; margin-top:8px; color:#94a3b8; font-size:0.85em;"># BEFORE
results = []
for img_id in images:
    results.append(compress_image(img_id))

# AFTER — swap these four lines for two</pre>
</div>
</details>

<details style="background:#0f172a; border:1px solid #475569; border-radius:8px; padding:10px 14px;">
<summary style="cursor:pointer; color:#94a3b8; font-weight:600; user-select:none; list-style:none;">
  🔑 Hint 6 — Full solution (try on your own first!)
</summary>
<div style="margin-top:10px; color:#cbd5e1; border-top:1px solid #1e293b; padding-top:10px; font-size:0.9em;">
  Replace the serial block with:
  <pre style="background:#020617; padding:10px; border-radius:6px; margin-top:8px; color:#e2e8f0; font-size:0.85em;">start = time.perf_counter()

with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(compress_image, images))

elapsed = time.perf_counter() - start</pre>
  That's it — two lines of parallel code replace four lines of serial code,
  and the checker at the bottom does the rest.
</div>
</details>

</div>

{% capture challenge_sc_d %}
Convert the serial `for` loop to parallel using `ThreadPoolExecutor`.
The checker at the bottom will automatically detect whether your code
runs faster than the serial baseline and report your speedup.
{% endcapture %}

{% capture code_sc_d %}
import time
from concurrent.futures import ThreadPoolExecutor

def compress_image(image_id):
    """Simulates compressing one image -- heavier for higher IDs."""
    result = sum(i * i for i in range(image_id * 40_000))
    return f"image_{image_id:03d}.jpg"

images = list(range(1, 9))  # 8 images to process

# ── TODO: convert the serial loop below to run in parallel ───────────────
# Hint: use ThreadPoolExecutor and pool.map()
#
#   with ThreadPoolExecutor(max_workers=4) as pool:
#       results = list(pool.map(compress_image, images))

start = time.perf_counter()

results = []
for img_id in images:           # <-- replace this loop with parallel code
    results.append(compress_image(img_id))

elapsed = time.perf_counter() - start

# ── CHECKER: do not modify below this line ───────────────────────────────
print("Processed:", ", ".join(r for r in results[:4]), "...")
print(f"Your time:  {elapsed:.3f}s")

_t = time.perf_counter()
for _i in images:
    compress_image(_i)
_serial = time.perf_counter() - _t

speedup = _serial / elapsed
print(f"Serial ref: {_serial:.3f}s")
print(f"Speedup:    {speedup:.1f}x")
print()
if speedup >= 2.0:
    print("PASSED -- parallelism detected. Nice work!")
elif speedup >= 1.3:
    print("PARTIAL -- some speedup, but try increasing max_workers.")
else:
    print("NOT YET -- still running serially. Replace the for loop:")
    print("  with ThreadPoolExecutor(max_workers=4) as pool:")
    print("      results = list(pool.map(compress_image, images))")
{% endcapture %}

{% include runners/code.html
   runner_id="sc-d"
   language="python"
   challenge=challenge_sc_d
   code=code_sc_d
   height="440px"
%}

<script>
function pdcCheck(qid, correct) {
  var sel = document.querySelector('input[name="'+qid+'"]:checked');
  var fb  = document.getElementById(qid+'-fb');
  var exp = document.getElementById(qid+'-exp');
  if (!sel) { if(fb){fb.textContent='Select an answer first!';fb.style.color='#f59e0b';} return; }
  var ok = sel.value===correct;
  if(fb){fb.innerHTML=ok?'✅ Correct!':'❌ Not quite — see below.';fb.style.color=ok?'#22c55e':'#ef4444';}
  if(exp) exp.style.display='block';
  document.querySelectorAll('input[name="'+qid+'"]').forEach(function(inp){
    var lbl=inp.parentElement;
    if(inp.value===correct){lbl.style.borderColor='#22c55e';lbl.style.background='#052e16';lbl.style.color='#86efac';}
    else if(inp===sel&&!ok){lbl.style.borderColor='#ef4444';lbl.style.background='#1c0606';lbl.style.color='#fca5a5';}
    inp.disabled=true; lbl.style.cursor='default';
  });
}
</script>

## Simulating the Difference

Build your own task queue, then watch all three approaches race.

- **Click** a task to increase its weight (1→2→…→7→1 dots)
- **Shift+click** or **drag** to multi-select — then click any selected task to cycle all at once
- **Deselect** to ungroup
- **Serial** — 1 worker, sequential &nbsp;|&nbsp; **Parallel** — 4 cores, no overhead &nbsp;|&nbsp; **Distributed** — 4 nodes, 600ms handshake but 1.5× faster processing. Try 1–2 dots (parallel wins) vs 3+ dots (distributed wins).

<div id="pdc-sim" style="font-family:sans-serif; max-width:960px; margin:auto;">

<!-- Config panel -->
<div style="background:#0f172a; border:1px solid #334155; border-radius:10px; padding:14px; margin-bottom:14px;">
  <div style="display:flex; gap:8px; flex-wrap:wrap; align-items:center; margin-bottom:12px;">
    <label style="color:#94a3b8; font-size:0.85em;">Tasks to add:
      <input id="pdc-n-input" type="number" min="1" value="4"
        style="width:52px; margin-left:6px; padding:4px 6px; border-radius:6px;
               background:#1e293b; color:#e2e8f0; border:1px solid #475569; font-size:0.95em;">
    </label>
    <button onclick="pdcAddTasks()"
      style="padding:5px 14px; border-radius:6px; border:none; background:#475569; color:#e2e8f0; cursor:pointer;">
      + Add
    </button>
    <button id="pdc-deselect-btn" onclick="pdcDeselect()"
      style="padding:5px 14px; border-radius:6px; border:none; background:#1e293b; color:#334155; cursor:pointer;">
      Deselect
    </button>
    <button onclick="pdcClearAll()"
      style="padding:5px 14px; border-radius:6px; border:none; background:#1e293b; color:#64748b; cursor:pointer;">
      Clear All
    </button>
    <span id="pdc-sel-label" style="color:#64748b; font-size:0.8em;"></span>
    <div style="margin-left:auto; display:flex; gap:8px; align-items:center;">
      <span style="color:#64748b; font-size:0.8em;">Speed:</span>
      <div id="pdc-speed-btns" style="display:flex; gap:4px;">
        <button onclick="pdcSetSpeed(1)"  data-speed="1"  style="padding:4px 8px; border-radius:5px; border:1px solid #3b82f6; background:#3b82f6; color:#fff; cursor:pointer; font-size:0.8em; font-weight:bold;">1×</button>
        <button onclick="pdcSetSpeed(2)"  data-speed="2"  style="padding:4px 8px; border-radius:5px; border:1px solid #334155; background:#1e293b; color:#94a3b8; cursor:pointer; font-size:0.8em;">2×</button>
        <button onclick="pdcSetSpeed(5)"  data-speed="5"  style="padding:4px 8px; border-radius:5px; border:1px solid #334155; background:#1e293b; color:#94a3b8; cursor:pointer; font-size:0.8em;">5×</button>
        <button onclick="pdcSetSpeed(10)" data-speed="10" style="padding:4px 8px; border-radius:5px; border:1px solid #334155; background:#1e293b; color:#94a3b8; cursor:pointer; font-size:0.8em;">10×</button>
        <button onclick="pdcSetSpeed(20)" data-speed="20" style="padding:4px 8px; border-radius:5px; border:1px solid #334155; background:#1e293b; color:#94a3b8; cursor:pointer; font-size:0.8em;">20×</button>
      </div>
      <button id="pdc-run-btn" onclick="pdcStart()"
        style="padding:6px 20px; border-radius:6px; border:none; background:#3b82f6; color:#fff; cursor:pointer; font-weight:bold;">
        ▶ Run
      </button>
      <button onclick="pdcReset()"
        style="padding:6px 14px; border-radius:6px; border:none; background:#334155; color:#cbd5e1; cursor:pointer;">
        ↺ Reset
      </button>
    </div>
  </div>
  <div id="pdc-task-wrap" style="position:relative; min-height:64px;">
    <div id="pdc-drag-rect" style="display:none; position:absolute; border:1px solid #3b82f6;
      background:rgba(59,130,246,0.1); pointer-events:none; border-radius:3px; z-index:10;"></div>
    <div id="pdc-task-grid" style="display:flex; flex-wrap:wrap; gap:6px; align-items:flex-start;">
      <span style="color:#334155; font-size:0.85em; align-self:center;">Add tasks above to get started</span>
    </div>
  </div>
  <div style="margin-top:8px; color:#475569; font-size:0.75em;">
    click to cycle weight &nbsp;|&nbsp; shift+click or drag to multi-select &nbsp;|&nbsp; ● = 1 work unit
  </div>
</div>

<!-- Simulation panels -->
<div style="display:flex; gap:12px; flex-wrap:wrap;">

  <div style="flex:1; min-width:220px; background:#0f172a; border:1px solid #334155; border-radius:10px; padding:12px;">
    <div style="font-weight:bold; color:#f59e0b; margin-bottom:8px;">Serial <span style="color:#475569; font-weight:normal; font-size:0.8em;">(1 worker)</span></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Queue</div>
    <div id="pdc-ser-queue" style="min-height:40px; margin-bottom:8px;"></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Worker</div>
    <div id="pdc-ser-worker" style="min-height:50px; margin-bottom:8px;"></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Done</div>
    <div id="pdc-ser-done" style="min-height:34px; margin-bottom:8px;"></div>
    <div id="pdc-ser-time" style="font-size:1.3em; font-weight:bold; color:#f59e0b;">—</div>
  </div>

  <div style="flex:1; min-width:220px; background:#0f172a; border:1px solid #334155; border-radius:10px; padding:12px;">
    <div style="font-weight:bold; color:#3b82f6; margin-bottom:8px;">Parallel <span style="color:#475569; font-weight:normal; font-size:0.8em;">(4 cores, 700ms/unit)</span></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Queue</div>
    <div id="pdc-par-queue" style="min-height:40px; margin-bottom:8px;"></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Workers</div>
    <div id="pdc-par-workers" style="min-height:50px; margin-bottom:8px;"></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Done</div>
    <div id="pdc-par-done" style="min-height:34px; margin-bottom:8px;"></div>
    <div id="pdc-par-time" style="font-size:1.3em; font-weight:bold; color:#3b82f6;">—</div>
  </div>

  <div style="flex:1; min-width:220px; background:#0f172a; border:1px solid #334155; border-radius:10px; padding:12px;">
    <div style="font-weight:bold; color:#22c55e; margin-bottom:8px;">Distributed <span style="color:#475569; font-weight:normal; font-size:0.8em;">(4 nodes, 450ms/unit + 📡600ms)</span></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Queue</div>
    <div id="pdc-dis-queue" style="min-height:40px; margin-bottom:8px;"></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Workers</div>
    <div id="pdc-dis-workers" style="min-height:50px; margin-bottom:8px;"></div>
    <div style="font-size:0.7em; color:#64748b; margin-bottom:3px; text-transform:uppercase;">Done</div>
    <div id="pdc-dis-done" style="min-height:34px; margin-bottom:8px;"></div>
    <div id="pdc-dis-time" style="font-size:1.3em; font-weight:bold; color:#22c55e;">—</div>
  </div>

</div>
<div id="pdc-result"></div>
</div>

<script>
(function () {
  const COLORS = [
    '#3b82f6','#8b5cf6','#ec4899','#f59e0b',
    '#10b981','#ef4444','#06b6d4','#f97316',
    '#a855f7','#14b8a6','#f43f5e','#84cc16',
    '#0ea5e9','#d946ef','#fb923c','#22d3ee'
  ];
  const TICK = 50;        // ms per simulation step — kept small for accuracy
  const TIME_UNIT = 700;
  const DIST_TIME_UNIT = 450;
  const NET_OVERHEAD = 600;
  const NUM_WORKERS = 4;

  let configTasks = [], nextId = 1, selection = new Set();
  let ser, par, dis, timer, running;
  let speedMult = 1;  // run this many simulation steps per visual frame

  // ── Drag-select state ───────────────────────────────────────
  let dragActive = false, dragMoved = false;
  let dragX0 = 0, dragY0 = 0, dragX1 = 0, dragY1 = 0;
  let suppressNextClick = false;

  function initDragSelect() {
    const wrap = document.getElementById('pdc-task-wrap');
    if (!wrap || wrap._dragBound) return;
    wrap._dragBound = true;
    wrap.addEventListener('mousedown', function (e) {
      if (running) return;
      const r = wrap.getBoundingClientRect();
      dragX0 = dragX1 = e.clientX - r.left;
      dragY0 = dragY1 = e.clientY - r.top;
      dragActive = true;
      dragMoved = false;
      e.preventDefault();
    });
  }

  document.addEventListener('mousemove', function (e) {
    if (!dragActive) return;
    const wrap = document.getElementById('pdc-task-wrap');
    if (!wrap) return;
    const r = wrap.getBoundingClientRect();
    dragX1 = e.clientX - r.left;
    dragY1 = e.clientY - r.top;
    if (Math.abs(dragX1 - dragX0) > 4 || Math.abs(dragY1 - dragY0) > 4) dragMoved = true;
    if (dragMoved) { updateDragRect(); updateDragSelection(wrap); }
  });

  document.addEventListener('mouseup', function () {
    if (!dragActive) return;
    dragActive = false;
    if (dragMoved) { suppressNextClick = true; hideDragRect(); }
  });

  function updateDragRect() {
    const el = document.getElementById('pdc-drag-rect');
    if (!el) return;
    const x1 = Math.min(dragX0, dragX1), y1 = Math.min(dragY0, dragY1);
    el.style.display = 'block';
    el.style.left   = x1 + 'px';
    el.style.top    = y1 + 'px';
    el.style.width  = Math.abs(dragX1 - dragX0) + 'px';
    el.style.height = Math.abs(dragY1 - dragY0) + 'px';
  }

  function hideDragRect() {
    const el = document.getElementById('pdc-drag-rect');
    if (el) el.style.display = 'none';
  }

  function updateDragSelection(wrap) {
    const wr = wrap.getBoundingClientRect();
    const rx1 = Math.min(dragX0, dragX1), ry1 = Math.min(dragY0, dragY1);
    const rx2 = Math.max(dragX0, dragX1), ry2 = Math.max(dragY0, dragY1);
    selection.clear();
    wrap.querySelectorAll('[data-task-id]').forEach(box => {
      const br = box.getBoundingClientRect();
      const bx1 = br.left - wr.left, by1 = br.top - wr.top;
      const bx2 = br.right - wr.left, by2 = br.bottom - wr.top;
      if (bx2 > rx1 && bx1 < rx2 && by2 > ry1 && by1 < ry2)
        selection.add(parseInt(box.dataset.taskId));
    });
    renderConfig();
  }

  // ── Speed ───────────────────────────────────────────────────
  window.pdcSetSpeed = function (n) {
    speedMult = n;
    document.querySelectorAll('#pdc-speed-btns button').forEach(btn => {
      const active = parseInt(btn.dataset.speed) === n;
      btn.style.background = active ? '#3b82f6' : '#1e293b';
      btn.style.color      = active ? '#fff'    : '#94a3b8';
      btn.style.border     = active ? '1px solid #3b82f6' : '1px solid #334155';
    });
  };

  // ── Config ──────────────────────────────────────────────────
  window.pdcAddTasks = function () {
    const n = Math.max(1, parseInt(document.getElementById('pdc-n-input').value) || 1);
    for (let i = 0; i < n; i++) {
      configTasks.push({ id: nextId, weight: 1, color: COLORS[(nextId - 1) % COLORS.length] });
      nextId++;
    }
    renderConfig();
  };

  window.pdcDeselect = function () { selection.clear(); renderConfig(); };

  window.pdcClearAll = function () {
    if (running) return;
    configTasks = []; nextId = 1; selection.clear();
    renderConfig(); clearPanels();
  };

  function renderConfig() {
    const grid = document.getElementById('pdc-task-grid');
    if (!configTasks.length) {
      grid.innerHTML = '<span style="color:#334155;font-size:0.85em;align-self:center;">Add tasks above to get started</span>';
      document.getElementById('pdc-sel-label').textContent = '';
      document.getElementById('pdc-deselect-btn').style.color = '#334155';
      return;
    }
    grid.innerHTML = configTasks.map(t => {
      const sel = selection.has(t.id);
      const dots = Array.from({length: 7}, (_, i) =>
        `<div style="width:7px;height:7px;border-radius:50%;
           background:${i < t.weight ? t.color : '#1e293b'};
           border:1px solid ${i < t.weight ? t.color : '#334155'};"></div>`
      ).join('');
      return `<div data-task-id="${t.id}" style="display:flex;flex-direction:column;align-items:center;
          gap:4px;padding:8px 6px;border-radius:8px;cursor:pointer;user-select:none;min-width:48px;
          background:${sel ? t.color+'22' : '#1e293b'};
          border:2px solid ${sel ? t.color : '#334155'};
          box-shadow:${sel ? '0 0 8px '+t.color+'55' : 'none'};">
        <span style="color:${t.color};font-weight:bold;font-size:12px;">T${t.id}</span>
        <div style="display:flex;flex-wrap:wrap;gap:2px;justify-content:center;max-width:28px;">${dots}</div>
        <span style="color:#64748b;font-size:10px;">${t.weight}u</span>
      </div>`;
    }).join('');

    const n = selection.size;
    document.getElementById('pdc-sel-label').textContent = n > 0 ? `${n} selected — click any to cycle all` : '';
    document.getElementById('pdc-deselect-btn').style.color = n > 0 ? '#e2e8f0' : '#334155';
  }

  document.addEventListener('click', function (e) {
    if (suppressNextClick) { suppressNextClick = false; return; }
    const box = e.target.closest('[data-task-id]');
    if (!box || running) return;
    const id = parseInt(box.dataset.taskId);
    if (e.shiftKey) {
      selection.has(id) ? selection.delete(id) : selection.add(id);
    } else if (selection.size > 0 && selection.has(id)) {
      selection.forEach(sid => {
        const t = configTasks.find(t => t.id === sid);
        if (t) t.weight = (t.weight % 7) + 1;
      });
    } else {
      const t = configTasks.find(t => t.id === id);
      if (t) t.weight = (t.weight % 7) + 1;
    }
    renderConfig();
  });

  // ── Simulation ──────────────────────────────────────────────
  function mkWorkers() {
    return Array.from({length: NUM_WORKERS}, () => ({task: null, phase: 'idle', elapsed: 0}));
  }

  function initSim() {
    const tasks = configTasks.map(t => ({...t}));
    ser = { queue: [...tasks], worker: null, elapsed: 0, done: [], ms: 0, finished: false };
    par = { queue: [...tasks], workers: mkWorkers(), done: [], ms: 0, finished: false };
    dis = { queue: [...tasks], workers: mkWorkers(), done: [], ms: 0, finished: false };
    document.getElementById('pdc-result').innerHTML = '';
    renderPanels();
  }

  function clearPanels() {
    ['ser','par','dis'].forEach(p => {
      ['queue','done'].forEach(s => { const el = document.getElementById(`pdc-${p}-${s}`); if (el) el.innerHTML = ''; });
      const t = document.getElementById(`pdc-${p}-time`); if (t) t.textContent = '—';
    });
    ['pdc-ser-worker','pdc-par-workers','pdc-dis-workers'].forEach(id => {
      const el = document.getElementById(id); if (el) el.innerHTML = '';
    });
  }

  // Each call advances the simulation by exactly TICK ms.
  // Speed is achieved by calling these multiple times per visual frame.
  function stepSerial() {
    if (ser.finished) return;
    ser.ms += TICK;
    if (!ser.worker && ser.queue.length > 0) { ser.worker = ser.queue.shift(); ser.elapsed = 0; }
    if (ser.worker) {
      ser.elapsed += TICK;
      if (ser.elapsed >= ser.worker.weight * TIME_UNIT) { ser.done.push(ser.worker); ser.worker = null; ser.elapsed = 0; }
    }
    if (!ser.worker && ser.queue.length === 0) ser.finished = true;
  }

  function stepWorkers(state, overhead, unitMs) {
    if (state.finished) return;
    state.ms += TICK;
    for (const w of state.workers) {
      if (w.phase === 'idle' && state.queue.length > 0) {
        w.task = state.queue.shift();
        w.phase = overhead ? 'connecting' : 'processing';
        w.elapsed = 0;
      }
      if (w.phase === 'connecting') {
        w.elapsed += TICK;
        if (w.elapsed >= NET_OVERHEAD) { w.phase = 'processing'; w.elapsed = 0; }
      } else if (w.phase === 'processing') {
        w.elapsed += TICK;
        if (w.elapsed >= w.task.weight * unitMs) {
          state.done.push(w.task); w.task = null; w.phase = 'idle'; w.elapsed = 0;
        }
      }
    }
    if (state.workers.every(w => w.phase === 'idle') && state.queue.length === 0) state.finished = true;
  }

  // Run speedMult simulation steps, then render once — accurate timing at all speeds.
  function step() {
    for (let i = 0; i < speedMult; i++) {
      stepSerial();
      stepWorkers(par, false, TIME_UNIT);
      stepWorkers(dis, true,  DIST_TIME_UNIT);
      if (ser.finished && par.finished && dis.finished) break;
    }
    renderPanels();
    if (ser.finished && par.finished && dis.finished) {
      clearInterval(timer); running = false;
      document.getElementById('pdc-run-btn').disabled = false;
      showResult();
    }
  }

  function taskBox(t, phase, elapsed, total) {
    const pct = phase === 'processing' && total > 0 ? Math.min(100, elapsed / total * 100) : 0;
    const connPct = phase === 'connecting' ? Math.min(100, elapsed / NET_OVERHEAD * 100) : 0;
    const c = phase === 'connecting' ? '#64748b' : t.color;
    return `<div style="display:inline-flex;flex-direction:column;align-items:center;
      background:${c}18;border:2px solid ${c};border-radius:6px;
      padding:3px 4px;margin:2px;min-width:36px;vertical-align:top;">
      <span style="color:${c};font-weight:bold;font-size:11px;">${phase==='connecting'?'📡':'T'+t.id}</span>
      <span style="color:#64748b;font-size:9px;">${t.weight}u</span>
      <div style="width:28px;height:4px;background:#1e293b;border-radius:2px;margin-top:2px;">
        <div style="width:${phase==='connecting'?connPct:pct}%;height:100%;background:${c};border-radius:2px;"></div>
      </div>
    </div>`;
  }

  function queueBox(t) {
    return `<div style="display:inline-flex;flex-direction:column;align-items:center;
      background:${t.color}12;border:1px solid ${t.color}55;border-radius:5px;
      padding:2px 5px;margin:1px;">
      <span style="color:${t.color};font-size:10px;font-weight:bold;">T${t.id}</span>
      <span style="color:#475569;font-size:9px;">${t.weight}u</span>
    </div>`;
  }

  function doneBox(t) {
    return `<div style="display:inline-flex;align-items:center;justify-content:center;
      background:${t.color}20;border:1px solid ${t.color}44;border-radius:5px;
      width:28px;height:28px;margin:1px;opacity:0.65;">
      <span style="color:${t.color};font-size:9px;font-weight:bold;">T${t.id}</span></div>`;
  }

  function idleSlot() {
    return `<div style="display:inline-flex;align-items:center;justify-content:center;
      width:40px;height:40px;border:1px dashed #334155;border-radius:6px;color:#334155;font-size:10px;">idle</div>`;
  }

  function renderPanels() {
    const empty = '<span style="color:#1e293b;font-size:11px;">—</span>';
    document.getElementById('pdc-ser-queue').innerHTML = ser.queue.length ? ser.queue.map(queueBox).join('') : empty;
    document.getElementById('pdc-ser-worker').innerHTML = ser.worker ? taskBox(ser.worker, 'processing', ser.elapsed, ser.worker.weight * TIME_UNIT) : idleSlot();
    document.getElementById('pdc-ser-done').innerHTML = ser.done.length ? ser.done.map(doneBox).join('') : empty;
    document.getElementById('pdc-ser-time').textContent = (ser.ms/1000).toFixed(1)+'s'+(ser.finished?' ✓':'');

    document.getElementById('pdc-par-queue').innerHTML = par.queue.length ? par.queue.map(queueBox).join('') : empty;
    document.getElementById('pdc-par-workers').innerHTML = par.workers.map((w,i) =>
      `<div style="display:flex;align-items:center;gap:4px;margin:1px 0;">
        <span style="color:#334155;font-size:10px;min-width:18px;">C${i+1}</span>
        ${w.task ? taskBox(w.task, w.phase, w.elapsed, w.task.weight * TIME_UNIT) : idleSlot()}
      </div>`).join('');
    document.getElementById('pdc-par-done').innerHTML = par.done.length ? par.done.map(doneBox).join('') : empty;
    document.getElementById('pdc-par-time').textContent = (par.ms/1000).toFixed(1)+'s'+(par.finished?' ✓':'');

    document.getElementById('pdc-dis-queue').innerHTML = dis.queue.length ? dis.queue.map(queueBox).join('') : empty;
    document.getElementById('pdc-dis-workers').innerHTML = dis.workers.map((w,i) =>
      `<div style="display:flex;align-items:center;gap:4px;margin:1px 0;">
        <span style="color:#334155;font-size:10px;min-width:18px;">N${i+1}</span>
        ${w.task ? taskBox(w.task, w.phase, w.elapsed, w.task.weight * DIST_TIME_UNIT) : idleSlot()}
      </div>`).join('');
    document.getElementById('pdc-dis-done').innerHTML = dis.done.length ? dis.done.map(doneBox).join('') : empty;
    document.getElementById('pdc-dis-time').textContent = (dis.ms/1000).toFixed(1)+'s'+(dis.finished?' ✓':'');
  }

  function showResult() {
    const parVsSer = (ser.ms/par.ms).toFixed(1);
    const disVsSer = (ser.ms/dis.ms).toFixed(1);
    const parWins = par.ms <= dis.ms;
    document.getElementById('pdc-result').innerHTML = `
      <div style="margin-top:14px;padding:12px;background:#0f172a;border:1px solid #334155;
        border-radius:8px;display:flex;gap:16px;flex-wrap:wrap;justify-content:center;align-items:center;">
        <span><span style="color:#94a3b8;">Serial: </span><strong style="color:#f59e0b;">${(ser.ms/1000).toFixed(1)}s</strong></span>
        <span style="color:#334155;">|</span>
        <span><span style="color:#94a3b8;">Parallel: </span><strong style="color:#3b82f6;">${(par.ms/1000).toFixed(1)}s</strong>
          <span style="color:#475569;font-size:0.85em;">(${parVsSer}× vs serial)</span></span>
        <span style="color:#334155;">|</span>
        <span><span style="color:#94a3b8;">Distributed: </span><strong style="color:#22c55e;">${(dis.ms/1000).toFixed(1)}s</strong>
          <span style="color:#475569;font-size:0.85em;">(${disVsSer}× vs serial)</span></span>
        <span style="background:#1e293b;padding:5px 10px;border-radius:6px;font-size:0.85em;">
          ${parWins
            ? `<span style="color:#3b82f6;">Parallel won</span> <span style="color:#64748b;">— overhead dominated light tasks</span>`
            : `<span style="color:#22c55e;">Distributed won</span> <span style="color:#64748b;">— raw speed beat the overhead</span>`}
        </span>
      </div>`;
  }

  window.pdcStart = function () {
    if (running || !configTasks.length) return;
    running = true;
    document.getElementById('pdc-run-btn').disabled = true;
    initSim();
    timer = setInterval(step, TICK);
  };

  window.pdcReset = function () {
    running = false; clearInterval(timer);
    document.getElementById('pdc-run-btn').disabled = false;
    document.getElementById('pdc-result').innerHTML = '';
    clearPanels();
  };

  renderConfig();
  initDragSelect();
})();
</script>

## TL;DR
- **Serial computing** executes tasks one after another on a single processor.
- **Parallel computing** divides tasks across multiple cores on the same machine, allowing simultaneous execution without network overhead.
- **Distributed computing** spreads tasks across multiple machines, which can provide more resources but incurs network communication overhead.
- The best approach depends on the problem size, resource availability, and performance requirements. For small tasks, parallel computing may outperform distributed due to lower overhead, while for larger tasks, distributed computing can leverage more resources for faster completion.